# Paired Comparison of the Frozen 7-Model Benchmark

## BRFSS 2025 Diabetes Classification

This notebook does **not** tune or redefine any model.

It reconstructs the frozen Part II data preparation and refits each of the seven frozen primary models once on the full training partition so that every model produces an aligned prediction for the **same 68,508 test respondents**.

The aligned predictions are then used for paired statistical comparisons.


## 1. Comparison Principles

The comparison is valid only if:

1. the same binary cohort and frozen 20 predictors are used;
2. the same deterministic 80/20 split is reconstructed;
3. each estimator exactly matches its frozen primary configuration;
4. reconstructed point metrics agree with the frozen model records within a small numerical tolerance;
5. no test result is used to change model parameters.

### Statistical analyses

- **Paired bootstrap** for ROC-AUC and PR-AUC among the four near-tied discrimination models: Logistic Regression, Gradient Boosting, XGBoost, and Linear SVM.
- **Exact McNemar tests** for classification disagreement between all model pairs at their frozen decision boundaries.
- **Holm correction** controls family-wise error across the 21 McNemar pairwise tests.

The paired bootstrap uses the same stratified resample for every model in each replicate.


In [ ]:
from pathlib import Path
from itertools import combinations
import glob
import json
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb

from scipy.stats import binomtest

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

OUT = Path("/kaggle/working/paired_model_comparison")
OUT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "DIABETE4"

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]
CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]
NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

xpt_candidates = glob.glob("/kaggle/input/**/LLCP2025.XPT", recursive=True)
if len(xpt_candidates) != 1:
    raise RuntimeError(f"Expected exactly one LLCP2025.XPT, found: {xpt_candidates}")
xpt_path = Path(xpt_candidates[0])

print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)


In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)
    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])

    out["_BMI5"] = raw["_BMI5"] / 100.0
    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)
    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)
    return out[PRIMARY_FEATURES]

target_mask = df[TARGET].isin([1.0, 3.0])
X = recode_brfss_2025(df.loc[target_mask, PRIMARY_FEATURES].copy())
y = (df.loc[target_mask, TARGET] == 1.0).astype("int8").rename("diabetes")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

assert len(X_train) == 274_031 and len(X_test) == 68_508
print("Train:", X_train.shape, "Test:", X_test.shape)


## 2. Fit the Frozen Preprocessing Once

Every frozen final-fit model used the same deterministic Part II transformer fitted on the same full training partition.

For this paired-comparison reconstruction, preprocessing is therefore fitted once and the identical transformed matrices are supplied to each frozen estimator. This is algebraically equivalent to fitting an identical deterministic transformer inside each model pipeline.


In [ ]:
categorical_pipeline = sklearn.pipeline.Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])
numeric_pipeline = sklearn.pipeline.Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
], remainder="drop")

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print("Transformed train:", X_train_t.shape)
print("Transformed test:", X_test_t.shape)


## 3. Refit the Seven Frozen Estimators

Configurations below are copied from the verified/frozen Model 01–07 specifications.

No parameter is selected in this notebook.


In [ ]:
models = {
    "logistic_regression": LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs", max_iter=1000, class_weight=None
    ),
    "decision_tree": DecisionTreeClassifier(
        criterion="gini", splitter="best", max_depth=None,
        min_samples_split=2, min_samples_leaf=1, max_features=None,
        class_weight=None, ccp_alpha=0.0, random_state=42
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100, criterion="gini", max_depth=None,
        min_samples_split=2, min_samples_leaf=1, max_features="sqrt",
        bootstrap=True, class_weight=None, ccp_alpha=0.0,
        random_state=42, n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingClassifier(
        loss="log_loss", learning_rate=0.1, n_estimators=100,
        subsample=1.0, criterion="friedman_mse",
        min_samples_split=2, min_samples_leaf=1, max_depth=3,
        max_features=None, random_state=42
    ),
    "xgboost": XGBClassifier(
        objective="binary:logistic", eval_metric="logloss", booster="gbtree",
        n_estimators=100, learning_rate=0.3, max_depth=6,
        min_child_weight=1, gamma=0.0, subsample=1.0,
        colsample_bytree=1.0, reg_alpha=0.0, reg_lambda=1.0,
        tree_method="hist", random_state=42, n_jobs=-1, verbosity=0
    ),
    "linear_svm": LinearSVC(
        penalty="l2", loss="squared_hinge", dual="auto",
        tol=1e-4, C=1.0, fit_intercept=True, class_weight=None,
        random_state=42, max_iter=10000
    ),
    "gaussian_nb": GaussianNB(priors=None, var_smoothing=1e-9),
}

predictions = pd.DataFrame({
    "respondent_index": X_test.index.to_numpy(),
    "true_label": y_test.to_numpy(),
})

dense_train = None
dense_test = None

for name, model in models.items():
    print("Fitting", name)
    if name == "gaussian_nb":
        if dense_train is None:
            dense_train = X_train_t.toarray() if hasattr(X_train_t, "toarray") else np.asarray(X_train_t)
            dense_test = X_test_t.toarray() if hasattr(X_test_t, "toarray") else np.asarray(X_test_t)
        model.fit(dense_train, y_train)
        score = model.predict_proba(dense_test)[:, 1]
        pred = (score >= 0.5).astype(int)
    elif name == "linear_svm":
        model.fit(X_train_t, y_train)
        score = model.decision_function(X_test_t)
        pred = model.predict(X_test_t)
    else:
        model.fit(X_train_t, y_train)
        score = model.predict_proba(X_test_t)[:, 1]
        pred = (score >= 0.5).astype(int)

    predictions[f"{name}_score"] = score
    predictions[f"{name}_pred"] = pred

predictions.to_csv(OUT / "aligned_test_predictions.csv", index=False)
print(predictions.shape)
display(predictions.head())


## 4. Reproduction Check Against Frozen Results

Before statistical testing, reconstructed test metrics must agree with the previously frozen model records.

A tolerance of `0.002` is used to catch meaningful reconstruction mismatches while allowing small library/runtime numerical differences.


In [ ]:
frozen = {
    "logistic_regression": {"accuracy":0.8558270567,"roc_auc":0.8261678123,"pr_auc":0.4466329084},
    "decision_tree": {"accuracy":0.7861563613,"roc_auc":0.6020699145,"pr_auc":0.2053377918},
    "random_forest": {"accuracy":0.8510392947,"roc_auc":0.8044953607,"pr_auc":0.4024650146},
    "gradient_boosting": {"accuracy":0.8563525428,"roc_auc":0.8260155802,"pr_auc":0.4483169950},
    "xgboost": {"accuracy":0.8548198751,"roc_auc":0.8262898794,"pr_auc":0.4417137783},
    "linear_svm": {"accuracy":0.8559876219,"roc_auc":0.8261580969,"pr_auc":0.4478840546},
    "gaussian_nb": {"accuracy":0.7272873241,"roc_auc":0.7885646729,"pr_auc":0.3641220118},
}

repro_rows = []
yt = predictions["true_label"].to_numpy()

for name in models:
    score = predictions[f"{name}_score"].to_numpy()
    pred = predictions[f"{name}_pred"].to_numpy()
    current = {
        "accuracy": accuracy_score(yt, pred),
        "roc_auc": roc_auc_score(yt, score),
        "pr_auc": average_precision_score(yt, score),
    }
    row = {"model": name}
    for metric in ["accuracy","roc_auc","pr_auc"]:
        row[f"reconstructed_{metric}"] = current[metric]
        row[f"frozen_{metric}"] = frozen[name][metric]
        row[f"abs_diff_{metric}"] = abs(current[metric] - frozen[name][metric])
    repro_rows.append(row)

reproduction_check = pd.DataFrame(repro_rows)
display(reproduction_check)
reproduction_check.to_csv(OUT / "reproduction_check.csv", index=False)

max_diff = reproduction_check.filter(like="abs_diff_").to_numpy().max()
print("Maximum absolute reproduction difference:", max_diff)
if max_diff > 0.002:
    raise RuntimeError(f"Frozen-model reproduction check failed: max difference {max_diff}")


## 5. Paired Bootstrap for Near-Tied Discrimination Models

The four models with ROC-AUC near 0.826 are compared using the same **1,000 stratified bootstrap samples**.

For each bootstrap sample we compute ROC-AUC and PR-AUC for every model. Pairwise differences therefore preserve respondent-level pairing.

A 95% percentile interval for a difference that includes zero indicates that the observed numerical ordering is not clearly separated by this bootstrap analysis.


In [ ]:
BOOTSTRAP_REPS = 1000
BOOTSTRAP_SEED = 20251010
top_models = ["logistic_regression", "gradient_boosting", "xgboost", "linear_svm"]

rng = np.random.default_rng(BOOTSTRAP_SEED)
pos_idx = np.flatnonzero(yt == 1)
neg_idx = np.flatnonzero(yt == 0)

boot_auc = {m: np.empty(BOOTSTRAP_REPS) for m in top_models}
boot_ap = {m: np.empty(BOOTSTRAP_REPS) for m in top_models}

scores = {m: predictions[f"{m}_score"].to_numpy() for m in top_models}

for b in range(BOOTSTRAP_REPS):
    s_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    s_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    idx = np.concatenate([s_pos, s_neg])
    yb = yt[idx]

    for m in top_models:
        sb = scores[m][idx]
        boot_auc[m][b] = roc_auc_score(yb, sb)
        boot_ap[m][b] = average_precision_score(yb, sb)

pair_rows = []
for a, b in combinations(top_models, 2):
    for metric_name, store in [("roc_auc", boot_auc), ("pr_auc", boot_ap)]:
        diff = store[a] - store[b]
        pair_rows.append({
            "model_a": a,
            "model_b": b,
            "metric": metric_name,
            "mean_difference_a_minus_b": float(diff.mean()),
            "ci95_lower": float(np.quantile(diff, 0.025)),
            "ci95_upper": float(np.quantile(diff, 0.975)),
            "proportion_diff_gt_zero": float(np.mean(diff > 0)),
        })

paired_bootstrap = pd.DataFrame(pair_rows)
display(paired_bootstrap)
paired_bootstrap.to_csv(OUT / "paired_bootstrap_top4.csv", index=False)


## 6. Exact McNemar Tests Across All Seven Models

McNemar's test compares paired classification disagreements on the same test respondents.

For each pair:
- `b` = A correct, B wrong;
- `c` = A wrong, B correct.

Under equal error probability among discordant cases, `b` is tested against Binomial(`b+c`, 0.5).

Because 21 pairwise tests are performed, p-values are adjusted with the Holm step-down procedure.


In [ ]:
model_names = list(models.keys())
correct = {
    m: predictions[f"{m}_pred"].to_numpy() == yt
    for m in model_names
}

mc_rows = []
for a, b in combinations(model_names, 2):
    ca, cb = correct[a], correct[b]
    n_a_correct_b_wrong = int(np.sum(ca & ~cb))
    n_a_wrong_b_correct = int(np.sum(~ca & cb))
    discordant = n_a_correct_b_wrong + n_a_wrong_b_correct

    if discordant == 0:
        p = 1.0
    else:
        p = float(binomtest(
            n_a_correct_b_wrong,
            n=discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue)

    mc_rows.append({
        "model_a": a,
        "model_b": b,
        "a_correct_b_wrong": n_a_correct_b_wrong,
        "a_wrong_b_correct": n_a_wrong_b_correct,
        "discordant_n": discordant,
        "p_exact": p,
    })

mcnemar = pd.DataFrame(mc_rows)

# Holm step-down adjusted p-values.
order = np.argsort(mcnemar["p_exact"].to_numpy())
raw = mcnemar["p_exact"].to_numpy()
adjusted = np.empty_like(raw)
running = 0.0
m = len(raw)
for rank, idx in enumerate(order):
    candidate = min(1.0, (m - rank) * raw[idx])
    running = max(running, candidate)
    adjusted[idx] = running

mcnemar["p_holm"] = adjusted
mcnemar["reject_0_05_holm"] = mcnemar["p_holm"] < 0.05

display(mcnemar.sort_values("p_holm"))
mcnemar.to_csv(OUT / "mcnemar_pairwise_holm.csv", index=False)


## 7. Consolidated ROC and Precision-Recall Curves

The curves below use the aligned predictions generated in this notebook. They are descriptive comparisons on the frozen internal test set.


In [ ]:
labels = {
    "logistic_regression":"Logistic Regression",
    "decision_tree":"Decision Tree",
    "random_forest":"Random Forest",
    "gradient_boosting":"Gradient Boosting",
    "xgboost":"XGBoost",
    "linear_svm":"Linear SVM",
    "gaussian_nb":"Gaussian NB",
}

plt.figure(figsize=(8,6))
for m in model_names:
    score = predictions[f"{m}_score"].to_numpy()
    fpr, tpr, _ = roc_curve(yt, score)
    auc = roc_auc_score(yt, score)
    plt.plot(fpr, tpr, label=f"{labels[m]} ({auc:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Frozen 7-Model ROC Comparison")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "roc_comparison.png", dpi=180)
plt.show()

plt.figure(figsize=(8,6))
for m in model_names:
    score = predictions[f"{m}_score"].to_numpy()
    precision, recall, _ = precision_recall_curve(yt, score)
    ap = average_precision_score(yt, score)
    plt.plot(recall, precision, label=f"{labels[m]} ({ap:.3f})")
plt.axhline(y=yt.mean(), linestyle="--", label=f"Prevalence ({yt.mean():.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Frozen 7-Model Precision-Recall Comparison")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "precision_recall_comparison.png", dpi=180)
plt.show()


## 8. Output Contract

This notebook exports:

- aligned per-respondent predictions;
- reproduction checks against frozen model metrics;
- paired bootstrap differences for the four near-tied discrimination models;
- exact pairwise McNemar tests with Holm correction;
- consolidated ROC and precision-recall figures.

These outputs support the final comparison narrative without modifying any primary model.
